# Monthly Water Demand Forecasting
This notebook explores a synthetic monthly demand series and compares a linear regression baseline with a seasonal-naive forecast. The data are illustrative, not utility records.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
 df_path = ROOT / 'data' / 'monthly_water_demand.csv'
df = pd.read_csv(df_path, parse_dates=['date']).sort_values('date').reset_index(drop=True)
df.head()

In [ ]:
df['t'] = np.arange(len(df))
m = df.date.dt.month
df['month_sin'] = np.sin(2*np.pi*m/12)
df['month_cos'] = np.cos(2*np.pi*m/12)
split = len(df)-12
train, test = df.iloc[:split], df.iloc[split:].copy()
features = ['t','month_sin','month_cos']
model = LinearRegression().fit(train[features], train.demand_ml)
test['regression'] = model.predict(test[features])
test['seasonal_naive'] = df.demand_ml.iloc[split-12:split].to_numpy()
for label, pred in [('Regression',test.regression),('Seasonal naive',test.seasonal_naive)]:
    print(label, 'MAE:', round(mean_absolute_error(test.demand_ml,pred),3),
          'RMSE:', round(np.sqrt(mean_squared_error(test.demand_ml,pred)),3))

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(train.date, train.demand_ml, label='Training data')
plt.plot(test.date, test.demand_ml, marker='o', label='Actual test data')
plt.plot(test.date, test.regression, marker='x', label='Regression forecast')
plt.xlabel('Month'); plt.ylabel('Demand (ML)'); plt.legend(); plt.tight_layout()

## Interpretation and limitations
The held-out year provides a simple check of forecast performance. With only five synthetic years, the exercise is illustrative. A real application would require verified utility data and explanatory variables such as weather, population, restrictions, leakage and industrial demand.